# Maize Leaf Anomaly Detection — Exploratory Notebook

This notebook provides an interactive walkthrough of the full pipeline:
- Dataset exploration
- Image visualisation
- Model training (or loading a pre-trained model)
- Evaluation and threshold selection
- Single-image inference demo

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

from src.config import *
from src.utils import set_seed
set_seed(RANDOM_SEED)

## 1. Dataset Overview

In [ ]:
from src.data_loader import discover_dataset, build_splits

class_map = discover_dataset(DATA_RAW_DIR)
for cls, paths in class_map.items():
    print(f'{cls}: {len(paths)} images')

In [ ]:
train_paths, val_paths, test_paths, test_labels = build_splits()
print(f'Train: {len(train_paths)}')
print(f'Val  : {len(val_paths)}')
print(f'Test : {len(test_paths)} ({sum(1 for l in test_labels if l==0)} healthy, {sum(test_labels)} diseased)')

## 2. Sample Images

In [ ]:
from src.preprocess import load_image

def show_samples(paths, title, n=8):
    fig, axes = plt.subplots(1, n, figsize=(2.5*n, 3))
    for i, ax in enumerate(axes):
        img = load_image(paths[i])
        ax.imshow(img)
        ax.axis('off')
    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

from src.config import HEALTHY_CLASS_NAME, DISEASED_CLASS_NAME
show_samples(class_map[HEALTHY_CLASS_NAME],  'Healthy Leaves')
show_samples(class_map[DISEASED_CLASS_NAME], 'Diseased Leaves')

## 3. Train or Load Model

In [ ]:
from pathlib import Path
from src.model import load_model, build_autoencoder

if Path(MODEL_PATH).exists():
    print('Loading pre-trained model …')
    model = load_model()
else:
    print('No saved model found — running training …')
    from src.train import train
    train()
    model = load_model()

## 4. Training History

In [ ]:
from src.utils import load_pickle
from src.train import plot_training_history

if Path(HISTORY_PATH).exists():
    history = load_pickle(HISTORY_PATH)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(history['loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(history['mae'], label='Train'); axes[1].plot(history['val_mae'], label='Val')
    axes[1].set_title('MAE'); axes[1].legend()
    plt.tight_layout(); plt.show()
else:
    print('No history file found.')

## 5. Reconstruction Examples

In [ ]:
from src.preprocess import load_images_from_paths
import numpy as np

n = 6
sample_paths = np.random.choice(class_map[HEALTHY_CLASS_NAME], size=n, replace=False)
sample_imgs  = load_images_from_paths(sample_paths)
recon_imgs   = model.predict(sample_imgs, verbose=0)

fig, axes = plt.subplots(2, n, figsize=(2.5*n, 5))
for i in range(n):
    axes[0,i].imshow(sample_imgs[i]); axes[0,i].axis('off')
    axes[1,i].imshow(np.clip(recon_imgs[i], 0, 1)); axes[1,i].axis('off')
axes[0,0].set_ylabel('Original',      fontsize=11)
axes[1,0].set_ylabel('Reconstructed', fontsize=11)
plt.suptitle('Healthy: Original vs Reconstructed', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 6. Evaluate & Select Threshold

In [ ]:
from src.evaluate import evaluate
metrics = evaluate()
print('\nMetrics:')
for k, v in metrics.items():
    print(f'  {k:<15}: {v}')

## 7. Single-Image Inference Demo

In [ ]:
from src.predict import AnomalyPredictor
import json

predictor = AnomalyPredictor()

# Pick a random healthy image
test_img = np.random.choice(class_map[HEALTHY_CLASS_NAME])
result   = predictor.predict(test_img)
print('Healthy test:', json.dumps(result, indent=2))

# Pick a random diseased image
test_img_d = np.random.choice(class_map[DISEASED_CLASS_NAME])
result_d   = predictor.predict(test_img_d)
print('Diseased test:', json.dumps(result_d, indent=2))